# Sieci konwolucyjne w PyTorch

In [ ]:
#!pip install tqdm

In [ ]:
import torch
import torchvision
import math
import numpy as np
from matplotlib import pyplot as plt

from tqdm import tqdm

%matplotlib inline
%config InlineBackend.figure_format = 'retina'
plt.rcParams["figure.figsize"] = [16, 9]

Akcelerator obliczeń.

In [ ]:
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f'Układ obliczeniowy: {device}')

## Zbiór danych

Przygotowujemy transformację wejściowych zdjęć. Obecnie jest to sekwencja dwóch operacji:
* zdekodowanie obrazu (`ToImage()`),
* konwersja do tensora PyTorch (`ToDtype()`)

In [ ]:
from torchvision import transforms

# Sekwencja operacji wykonujących pre-processing danych wejściowych.
input2tensor = transforms.Compose([
    transforms.ToTensor(),
])

Ładujemy dane i wyświetlamy kila przykładowych obrazów.

In [ ]:
train_data = torchvision.datasets.FashionMNIST('/tmp/fmnist', transform=input2tensor,
                                               train=True, download=True)
test_data = torchvision.datasets.FashionMNIST('/tmp/fmnist', transform=input2tensor,
                                              train=False, download=True)

In [ ]:
subset = np.random.choice(np.arange(60000), size=16)
plt.figure(figsize=(10,10))
for i in range(16):
    plt.subplot(4,4,i+1)
    plt.xticks([])
    plt.yticks([])
    plt.grid(False)
    img, label = train_data[subset[i]]
    plt.imshow(img.squeeze(), cmap=plt.cm.binary)

plt.show()

In [ ]:
from torch.utils.data import DataLoader

train_dataloader = DataLoader(train_data, batch_size=256, num_workers=4, shuffle=True)
test_dataloader = DataLoader(test_data, batch_size=256, num_workers=4, shuffle=True)

Funkcje do trenowania i ewaluacji sieci.

In [ ]:
def train_network(model, loss_f, opt, train_dataloader, epoch_num):
    # Wpierw przełączamy sieć w tryb uczenia. Tą operację należy wykonać
    # przed każdą sesją trenowania.
    model.train()

    # Pętla po epokach
    for epoch in range(epoch_num):
        epoch_loss, batch_count = 0, 0

        # Pętla po mini-batchach z modułu ładującego dane
        for images, labels in tqdm(train_dataloader):
            # Zerujemy pochodne w algorytmie optymalizacji.
            # Poprzednio operację tą trzeba było wykonać ręcznie.
            opt.zero_grad()

            # Wysyłamy przykłady uczące na kartę graficzną
            images = images.to(device)
            labels = labels.to(device)

            # Operacja feed-forward - wynikiem są logity
            logits = model(images)

            # Teraz możemy obliczyć koszt i wykonać wsteczną propagację błędów
            loss = loss_f(logits, labels)
            loss.backward()

            # Aktualizacja wartości parametrów. W pakiecie optim wystarczy w
            # tym celu wykonać metodę 'step()'
            opt.step()

            epoch_loss += loss.item()
            batch_count += 1

        print(f'Epoka {epoch+1} koszt na zbiorze uczącym: {epoch_loss / batch_count}')

In [ ]:
def evaluate_network(model, loss_f, test_dataloader):
    test_loss, batch_count = 0, 0
    correct_predictions  = 0
    example_count = 0

    # Aby móc użyć wytrenowany model musimy przełączyć go w tryb ewaluacji.
    # Operacją tą należy wykonać zanim sieć zostanie wykorzystana do predykcji.
    model.eval()

    # Pętla po przykładach testowych
    for images, labels in tqdm(test_dataloader):

        # Wysyłamy przykłady na kartę graficzną
        images = images.to(device)
        labels = labels.to(device)

        # Operacja feed-forward
        logits = model(images)

        # Jako przewidywaną klasę przyjmujemy klasę z największym logitem
        _, predicted_class = torch.max(logits, 1)

        # Zliczamy prawidłowe predykcje
        correct_predictions += (predicted_class == labels).sum().item()
        example_count += len(predicted_class)

        # Obliczamy wartość kosztu na zbiorze testowym
        loss = loss_f(logits, labels)
        test_loss += loss.item()
        batch_count += 1

    print(f'Dokładność na zbiorze testowym: {correct_predictions / example_count*100:.2f}%')
    print(f'Koszt na zbiorze testowym: {test_loss / batch_count:.2f}')

## Zadanie

Zaimplementuj sieć konwolucyjną do klasyfikacji obrazów ze zbioru `FashionMNIST`. Wynik zapisz w zmiennej `cnn_model`.

W pakiecie PyTorch warstwa konwolucyjna implementowana jest przez metodę `torch.nn.Conv2d`. Implementując sieć posłuż się dokumentacją tej metody. Sieć powinna składać się z:
* Dwóch warstw konwolucyjnych, z których pierwsza ma 4 kanały wyjściowe a druga 32 kanały wyjściowe. Obie warstwy powinny mieć kernele o rozmiarze $(3, 3)$ przesuwane w obu osiach co dwa piksele. Warstwy nie powinny mieć paddingu.
* Warstwy w pełni połączonej wyliczającej logity.

Ponadto:
* Po każdej warstwie konwolucyjnej powinna być dodana warstwa implementująca aktywację `ReLU` oraz warstwa implementująca operację normalizacji aktywacji (`torch.nn.BatchNorm2d`).
* Przed warstwą w pełni połączoną należy dodać warstwę rzutującą wolumen konwolucyjny na wektor liczb.

Zwróć uwagę, że wejściowe obrazy mają jeden kanał (odcienie szarości).

In [ ]:
# Implementacja sieci konwolucyjnej
cnn_model = torch.nn.Sequential(
    # Dokończ zgodnie z poleceniem implementację sieci konwolucyjnej.
    
)

Wysłamy sieć na GPU.

In [ ]:
cnn_model.to(device)

Funkcja kosztu i algorytm optymalizacji.

In [ ]:
loss_f = torch.nn.CrossEntropyLoss(reduction='mean')

In [ ]:
opt = torch.optim.SGD(cnn_model.parameters(), lr=0.01, momentum=0.9)

Trenujemy sieć.

In [ ]:
train_network(cnn_model, loss_f, opt, train_dataloader, 20)

I oceniamy jej skuteczność.

In [ ]:
evaluate_network(cnn_model, loss_f, test_dataloader)